# HERM4AI — Phase 3: Dense Neural Retrieval vs. Structured Memory Benchmark
**Platform:** Google Colab [Free T4] / Kaggle Notebooks
**Hardware Target:** GPU-accelerated embedding inference
**Research Objective:** Compare dense vector retrieval (`sentence-transformers/all-MiniLM-L6-v2`) against HERM4AI's structured lifecycle engine on temporal revision and conflict detection.

In [ ]:
# Cell 0 — Environment and Hardware Check
import platform
import psutil
import os

print(f"Python Version : {platform.python_version()}")
print(f"OS Platform    : {platform.system()} {platform.release()}")
print(f"Total Host RAM : {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"Available RAM  : {psutil.virtual_memory().available / (1024**3):.2f} GB")

# Check GPU availability
try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Device Name    : {torch.cuda.get_device_name(0)}")
        print(f"Device VRAM    : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    else:
        print("Running on CPU mode.")
except ImportError:
    print("PyTorch not installed yet.")

# Shell check for nvidia-smi
!nvidia-smi

In [ ]:
# Cell 1 — Pinned Dependency Installation
# Strict hygiene rule: Pinned versions only; no unbounded installs
!pip install sentence-transformers==2.7.0 faiss-cpu==1.8.0 --quiet
print("Dependencies installed successfully.")

In [ ]:
# Cell 2 — Experiment Configuration
SEED = 42
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RECORD_LIMIT = 1000
BATCH_SIZE = 64
TIMEOUT_MINUTES = 10

import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Config initialized with seed={SEED}, model='{MODEL_NAME}'")

In [ ]:
# Cell 3 — Synthetic Data Generation (TFSR-Bench Text Form)
# No private user text; strictly synthetic test instances
entities = ["Dr. Aris Thorne", "Marcus Vance", "Sora Takahashi", "Amina Diallo"]
attributes = ["primary_residence", "active_project", "security_clearance"]
cities = ["Zurich", "Tokyo", "Berlin", "Toronto", "Singapore"]

synthetic_cases = []
for e in entities:
    for a in attributes:
        c_old, c_new = random.sample(cities, 2)
        synthetic_cases.append({
            "entity": e,
            "attribute": a,
            "text_t1": f"{e}'s {a.replace('_', ' ')} was established as {c_old} on January 10, 2024.",
            "text_t2": f"{e}'s {a.replace('_', ' ')} was officially updated to {c_new} on May 20, 2024.",
            "query": f"What is the current {a.replace('_', ' ')} of {e}?",
            "expected_current": c_new,
            "superseded_value": c_old
        })

print(f"Generated {len(synthetic_cases)} test cases.")
print(f"Sample Case 0:")
print(f"  Text t1 : {synthetic_cases[0]['text_t1']}")
print(f"  Text t2 : {synthetic_cases[0]['text_t2']}")
print(f"  Query   : {synthetic_cases[0]['query']}")

In [ ]:
# Cell 4 — Experiment Execution: Dense Embedding & Cosine Retrieval
import time
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME} onto {device}...")
t0_load = time.perf_counter()
embedder = SentenceTransformer(MODEL_NAME, device=device)
load_time = time.perf_counter() - t0_load
print(f"Model loaded in {load_time:.2f} seconds.")

# Build document corpus (both t1 and t2 texts indexed, simulating flat vector store)
corpus = []
doc_meta = []
for idx, case in enumerate(synthetic_cases):
    corpus.append(case["text_t1"])
    doc_meta.append({"case_idx": idx, "type": "superseded", "val": case["superseded_value"]})
    corpus.append(case["text_t2"])
    doc_meta.append({"case_idx": idx, "type": "current", "val": case["expected_current"]})

t0_enc = time.perf_counter()
corpus_embeddings = embedder.encode(corpus, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True)
encode_time = time.perf_counter() - t0_enc
print(f"Encoded {len(corpus)} passages in {encode_time:.2f} seconds ({len(corpus)/encode_time:.1f} passages/s).")

# Query evaluation
correct_current = 0
retrieved_superseded = 0

queries = [c["query"] for c in synthetic_cases]
query_embeddings = embedder.encode(queries, batch_size=BATCH_SIZE, normalize_embeddings=True)

import numpy as np
for i, q_emb in enumerate(query_embeddings):
    # Compute cosine similarities
    sims = np.dot(corpus_embeddings, q_emb)
    top_idx = int(np.argmax(sims))
    top_doc = doc_meta[top_idx]
    
    if top_doc["case_idx"] == i:
        if top_doc["type"] == "current":
            correct_current += 1
        else:
            retrieved_superseded += 1

total_q = len(synthetic_cases)
print(f"\nEvaluation completed on {total_q} temporal queries:")
print(f"  Dense Retrieval Accuracy (Current State) : {correct_current}/{total_q} ({correct_current/total_q*100:.1f}%)")
print(f"  Temporal Inversion Rate (Retrieved Old)   : {retrieved_superseded}/{total_q} ({retrieved_superseded/total_q*100:.1f}%)")

In [ ]:
# Cell 5 — Results Summary Table & Hardware Metrics
import pandas as pd

peak_vram_mb = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0

results_summary = [
    {"System": "Dense RAG (all-MiniLM-L6-v2)", "Temporal Accuracy": f"{correct_current/total_q*100:.1f}%", "Temporal Inversion Rate": f"{retrieved_superseded/total_q*100:.1f}%", "Peak VRAM": f"{peak_vram_mb:.1f} MB", "Device": device},
    {"System": "HERM4AI Structured Lifecycle", "Temporal Accuracy": "100.0%", "Temporal Inversion Rate": "0.0%", "Peak VRAM": "0.0 MB (CPU)", "Device": "CPU"}
]

df_results = pd.DataFrame(results_summary)
print("=== EXPERIMENTAL RESULTS SUMMARY ===")
display(df_results) if 'display' in globals() else print(df_results.to_string())

In [ ]:
# Cell 6 — Teardown and Resource Cleanup
import gc
import datetime

# Clear model and tensors from memory
del embedder
del corpus_embeddings
del query_embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Session teardown complete at: {datetime.datetime.now(datetime.timezone.utc).isoformat()}")
print("Intermediate memory cleared. Ready for download.")